In [7]:
import pandas as pd

# 1. Load the raw file
df = pd.read_csv("hersh_exome_biallelic_annovar_annotated.hg38_multianno.indels.first_10000.txt", sep="\t", low_memory=False)
print("total rows:", len(df))
print("total cols: ", len(df.columns))
print(df.columns)

# 2. keep only exonic variants
exonic = df[df["Func.refGeneWithVer"] == "exonic"].copy()
print("exonic rows:", len(exonic))
print(exonic["ExonicFunc.refGeneWithVer"].value_counts())

# 3. Pick the score columns (drop metadata)
meta = ("Chr","Start","End","Ref","Alt","Func.","Gene.","GeneDetail.","ExonicFunc.",
        "AAChange.","gnomad","CLN","ONC","SCI","Interpro","GTEx","eQTL","Otherinfo","REGENERON")
score_cols = [c for c in exonic.columns if not c.startswith(meta)]
print(len(score_cols), "score columns")

# 4. Does each tool have a score for each variant?
has_score = pd.DataFrame({
    c: ~exonic[c].astype(str).str.strip().isin([".", "", "nan", "NaN", "None"])
    for c in score_cols
}, index=exonic.index)
has_score["ExonicFunc"] = exonic["ExonicFunc.refGeneWithVer"].values

# 5. Fraction covered: tools as rows, variant types as columns
coverage = has_score.groupby("ExonicFunc")[score_cols].mean().round(3).T
print(coverage)
coverage.to_csv("tool_variant_coverage.csv")

# 6. which tools score indels?
indel_cols = [c for c in ["frameshift deletion","frameshift insertion",
                          "nonframeshift deletion","nonframeshift insertion"]
              if c in coverage.columns]
scores_indels = coverage[coverage[indel_cols].max(axis=1) > 0]
print("\nTools that score at least some indels:")
print(scores_indels[indel_cols])

total rows: 10000
total cols:  229
Index(['Chr', 'Start', 'End', 'Ref', 'Alt', 'Func.refGeneWithVer',
       'Gene.refGeneWithVer', 'GeneDetail.refGeneWithVer',
       'ExonicFunc.refGeneWithVer', 'AAChange.refGeneWithVer',
       ...
       'Otherinfo2', 'Otherinfo3', 'Otherinfo4', 'Otherinfo5', 'Otherinfo6',
       'Otherinfo7', 'Otherinfo8', 'Otherinfo9', 'Otherinfo10', 'Otherinfo11'],
      dtype='str', length=229)
exonic rows: 62
ExonicFunc.refGeneWithVer
frameshift deletion        27
frameshift insertion       15
nonframeshift insertion     9
nonframeshift deletion      9
stopgain                    2
Name: count, dtype: int64
144 score columns
ExonicFunc                      frameshift deletion  frameshift insertion  \
SIFT_score                                      0.0                   0.0   
SIFT_converted_rankscore                        0.0                   0.0   
SIFT_pred                                       0.0                   0.0   
SIFT4G_score                     

In [3]:
print(len(df), "rows")
print(df["Func.refGeneWithVer"].value_counts().head())

10000 rows
Func.refGeneWithVer
intronic      8847
intergenic     533
UTR3           421
upstream        82
exonic          62
Name: count, dtype: int64


In [4]:
ref = df["Ref"].astype(str).str.strip()
alt = df["Alt"].astype(str).str.strip()
is_indel = (ref == "-") | (alt == "-") | (ref.str.len() != alt.str.len())
print(is_indel.value_counts())

True    10000
Name: count, dtype: int64


In [ ]:
# 1. CADD columns
cadd_cols = [c for c in df.columns if "CADD" in c]
print("CADD columns:", cadd_cols)

# 2. How many rows in the WHOLE file have a CADD value?
print("\nAcross all", len(df), "rows:")
for c in cadd_cols:
    s = df[c].astype(str).str.strip()
    n = (~s.isin([".", "", "nan", "NaN", "None"])).sum()
    print(f"  {c}: {n} rows have a value")

# 3.just the exonic indels
print("\nAcross the", len(exonic), "exonic rows:")
for c in cadd_cols:
    s = exonic[c].astype(str).str.strip()
    n = (~s.isin([".", "", "nan", "NaN", "None"])).sum()
    print(f"  {c}: {n} rows have a value")

CADD columns: ['CADD_raw', 'CADD_raw_rankscore', 'CADD_phred']

Across all 10000 rows:
  CADD_raw: 0 rows have a value
  CADD_raw_rankscore: 0 rows have a value
  CADD_phred: 0 rows have a value

Across the 62 exonic rows:
  CADD_raw: 0 rows have a value
  CADD_raw_rankscore: 0 rows have a value
  CADD_phred: 0 rows have a value

Sample of raw CADD values (exonic):
   Ref                    Alt ExonicFunc.refGeneWithVer CADD_raw CADD_raw_rankscore CADD_phred
61   -                CAGGCGG      frameshift insertion        .                  .          .
62   -                GGGTCGT      frameshift insertion        .                  .          .
63   -              GGGTCGTTA   nonframeshift insertion        .                  .          .
64   -                 TCGTTC                  stopgain        .                  .          .
65   -  GTTGTCCAAACGTTCGTTGGT   nonframeshift insertion        .                  .          .
66   -                   GCGG      frameshift insertion       